# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and processing the FAIR^2 dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library. The approach closely follows best practices in transparent, reproducible data science and relies on referencing entities by their schema `@id`.

### Dataset Source
The dataset is publicly accessible via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

The dataset covers survey-based adoption predictors of indigenous and modern knowledge in rangeland management practices among pastoral communities in Northern Kenya.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Initialize Dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and see which fields and columns each contains. These will be referenced by `@id` for subsequent data extraction and analyses.

In [ ]:
# List all record sets by `@id`
if hasattr(dataset, 'record_sets'):
    print(f"Found {len(dataset.record_sets)} record sets.")
    for rs in dataset.record_sets:
        # Each record set is a RecordSet object
        print(f"- Record set name: {rs.name}, @id: {rs.id}")
        # Field overview
        if hasattr(rs, 'fields'):
            print("    Fields:")
            for field in rs.fields:
                print(f"      - Field: {getattr(field,'name',None)} (@id: {getattr(field,'id',None)})")
        # Column overview (csv):
        if hasattr(rs, 'columns') and rs.columns:
            print("    Columns:")
            for col in rs.columns:
                print(f"      - Column: {getattr(col,'name',None)} (@id: {getattr(col,'id',None)})")

else:
    print("No record sets found in this dataset. Check dataset structure.")

## 3. Data Extraction

Using the above record set `@id`s (replace the examples below with real values), extract data.

*You must reference all record sets and fields by their `@id`. If you need to change which record set to work with, set `record_sets` and `main_record_set_id` variables accordingly.*

In [ ]:
# ---- Set main record set IDs here ----
# Here we demonstrate with all available record sets, if any.

# Gather all record set @ids
record_sets = [rs.id for rs in getattr(dataset, 'record_sets', [])]
print(f"Extracting data from record sets by @id: {record_sets}")

dataframes = {}
for record_set_id in record_sets:
    # Records is a generator of dicts
    print(f"Loading data for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only create DataFrame if there are records
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for record set {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for record set {record_set_id}.")

# For subsequent analysis, pick one main record set if present
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    print(f"Main record set selected: {main_record_set_id}")
else:
    print('No record sets available for further analysis.')

## 4. Exploratory Data Analysis (EDA)
Typical EDA steps: filtering, normalization, and grouping will be performed by referencing columns *strictly by their `@id` values* as listed above. Here, choose a numeric field and a grouping field as demonstrated.

In [ ]:
import numpy as np

if main_record_set_id and main_record_set_id in dataframes and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]

    print(f"Columns available (referenced by @id):\n{df.columns.tolist()}")
    # Example: pick a numeric and group field (modify according to your columns' @id)
    potential_numeric_fields = [c for c in df.columns if df[c].dtype in [np.float64, np.float32, np.int64, np.int32]]
    if not potential_numeric_fields:
        # Try to convert any column to numeric if possible
        for c in df.columns:
            try:
                converted = pd.to_numeric(df[c], errors='coerce')
                if converted.notnull().mean() > 0.8:  # At least 80% numeric
                    df[c] = converted
                    potential_numeric_fields.append(c)
            except Exception:
                pass

    if potential_numeric_fields:
        numeric_field = potential_numeric_fields[0]  # Choose first numeric field
        print(f"Using numeric field (by @id): {numeric_field}")
    else:
        print("No suitable numeric field found for further analysis.")
        numeric_field = None

    # Select a group field (categorical, not numeric)
    group_fields = [c for c in df.columns if c != numeric_field and df[c].nunique() < (0.1*len(df))]
    group_field = group_fields[0] if group_fields else None
    if group_field:
        print(f"Grouping by field (by @id): {group_field}")

    # EDA example: filter, normalize, group
    if numeric_field:
        # Drop NA for robust stats
        filtered_df = df[df[numeric_field].notna()]
        threshold = filtered_df[numeric_field].mean()  # Example threshold: mean
        filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} records.")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field}, showing mean of {numeric_field} per group:")
            display(grouped_df.head())
    else:
        print("No numeric field to process for EDA.")
else:
    print("No data available in main record set for EDA.")

## 5. Visualization
Visualize selected distributions or relationships using the normalized numeric field and group field, if available.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if main_record_set_id and main_record_set_id in dataframes and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]
    # Use columns from previous cell (numeric_field, group_field)
    if 'filtered_df' in locals() and numeric_field:
        plt.figure(figsize=(8,5))
        plt.hist(filtered_df[numeric_field], bins=25, alpha=0.7, color='teal')
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()

        if group_field:
            plt.figure(figsize=(10,6))
            filtered_df.boxplot(column=numeric_field, by=group_field, grid=False, rot=45)
            plt.title(f"{numeric_field} by {group_field}")
            plt.suptitle("")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion

This notebook demonstrates how to explore, process, and visualize a Croissant-annotated dataset using `mlcroissant` strictly referencing entities by their schema `@id`. The flexible workflow allows you to inspect record sets, select fields dynamically, apply transformations, and visualize relationships for informed, reproducible data science on FAIR^2 datasets.

For further use, customize the analysis sections to your research questions and make use of field and record set `@id`s as the single source of truth for programmatic access.